In [4]:
import os
import cv2
import mediapipe as mp
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# -----------------------------
# Initialize MediaPipe Pose
# -----------------------------


ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
mp_pose = mp.solutions.pose

pose = mp_pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.5
)

# -----------------------------
# Dataset Folder
# -----------------------------
dataset_path = "Final_datasets/Train"

# Store all samples
data = []

In [ ]:
# -----------------------------
# Read every pose folder
# -----------------------------
for pose_name in os.listdir(dataset_path):

    pose_folder = os.path.join(dataset_path, pose_name)

    if not os.path.isdir(pose_folder):
        continue

    print(f"Processing {pose_name}")

    # Read every image
    for image_name in os.listdir(pose_folder):

        image_path = os.path.join(pose_folder, image_name)

        image = cv2.imread(image_path)

        if image is None:
            continue

        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        results = pose.process(rgb)

        if not results.pose_landmarks:
            continue

        landmarks = []

        # -----------------------------
        # Extract landmarks
        # -----------------------------
        for landmark in results.pose_landmarks.landmark:

            landmarks.append(landmark.x)
            landmarks.append(landmark.y)
            landmarks.append(landmark.visibility)

            # If your project used only X,Y
            # remove the visibility line.

        # Add Label
        landmarks.append(pose_name)

        data.append(landmarks)



In [ ]:
# -----------------------------
# Create DataFrame
# -----------------------------

columns = []

for i in range(33):
    columns.append(f"x{i}")
    columns.append(f"y{i}")
    columns.append(f"v{i}")

columns.append("label")

df = pd.DataFrame(data, columns=columns)

# Save CSV
df.to_csv("pose_dataset.csv", index=False)

print("CSV Created Successfully")



In [ ]:
# -----------------------------
# Read CSV
# -----------------------------

dataset = pd.read_csv("pose_dataset.csv")

# Features

X = dataset.drop("label", axis=1)

# Labels

y = dataset["label"]

# -----------------------------
# Split Dataset
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -----------------------------
# Create Random Forest
# -----------------------------

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# -----------------------------
# Train Model
# -----------------------------

model.fit(X_train, y_train)

print("Training Completed")

# -----------------------------
# Test Model
# -----------------------------

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy :", accuracy)

# -----------------------------
# Save Model
# -----------------------------

joblib.dump(model, "pose_classifier_model.joblib")

print("Model Saved Successfully")